In [1]:
import pandas as pd
import fastf1
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

DIR_RAW = Path('../data/raw')
DIR_PROCESSED = Path('../data/processed')

YEAR = 2025
RACES = 24

fastf1.Cache.enable_cache(DIR_RAW)


In [2]:
def time_to_str(time:str) -> str:

    """Remove os primeiros 6 caracteres do timedelta string"""

    return time[6:]

In [3]:
def extract(num_race:int) -> pd.DataFrame:

    session = fastf1.get_session(YEAR, num_race, 'Q')
    session.load()
    laps = session.laps
    race = session.event['EventName']
    laps['Race'] = race
    laps['RaceNumber'] = num_race

    return laps

In [5]:
def get_perfect_lap(laps: pd.DataFrame) -> pd.DataFrame:

    fastest_laps = (laps.groupby(by=['Driver'], as_index=False)
                    ['LapTime'].min()
                    .sort_values(by=['LapTime'])
                    )
    
    sectors = laps[['Driver', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Race', 'RaceNumber']]
    fastest_sectors = sectors.groupby(by=['Driver'], as_index=False).min()

    bestS1 = sectors['Sector1Time'].min()
    bestS2 = sectors['Sector2Time'].min()
    bestS3 = sectors['Sector3Time'].min()

    perfect_lap = bestS1 + bestS2 + bestS3

    fastest_laps_sectors = fastest_laps.merge(right=fastest_sectors,
                                              how='left',
                                              on='Driver')
    
    fastest_laps_sectors['IdealLap'] = perfect_lap

    return fastest_laps_sectors   
    


In [6]:
def diff_time(fastest_laps_sectors: pd.DataFrame) -> pd.DataFrame:

    fastest_laps_sectors['DiffLap'] = fastest_laps_sectors['LapTime'] - fastest_laps_sectors['IdealLap']

    return fastest_laps_sectors

In [7]:
def diff_sec(fastest_laps_sectors: pd.DataFrame) -> pd.DataFrame:
    
    fastest_laps_sectors['LapTimeSec'] = fastest_laps_sectors['LapTime'].dt.total_seconds()
    fastest_laps_sectors['IdealLapSec'] = fastest_laps_sectors['IdealLap'].dt.total_seconds()
    fastest_laps_sectors['DiffLapSec'] = fastest_laps_sectors['DiffLap'].dt.total_seconds()

    return fastest_laps_sectors

In [9]:
def fix_time(fastest_laps_sectors: pd.DataFrame) -> pd.DataFrame:
    
    fastest_laps_sectors['LapTime'] = fastest_laps_sectors['LapTime'].astype(str).apply(time_to_str)
    fastest_laps_sectors['Sector1Time'] = fastest_laps_sectors['Sector1Time'].astype(str).apply(time_to_str)
    fastest_laps_sectors['Sector2Time'] = fastest_laps_sectors['Sector2Time'].astype(str).apply(time_to_str)
    fastest_laps_sectors['Sector3Time'] = fastest_laps_sectors['Sector3Time'].astype(str).apply(time_to_str)
    fastest_laps_sectors['IdealLap'] = fastest_laps_sectors['IdealLap'].astype(str).apply(time_to_str)
    fastest_laps_sectors['DiffLap'] = fastest_laps_sectors['DiffLap'].astype(str).apply(time_to_str)

    return fastest_laps_sectors

In [ ]:
def organize_columns(df: pd.DataFrame) -> pd.DataFrame:

    new_order = ['Driver', 'LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'IdealLap', 'DiffLap', 'LapTimeSec', 'IdealLapSec', 'DiffLapSec', 'Race', 'RaceNumber']
    df = df[new_order]

    return df

In [ ]:
def data_transformation():
    
    for race in range(1, 25):
        laps = extract(race)
        df = get_perfect_lap(laps)
        df = diff_time(df)
        df = diff_sec(df)
        df = fix_time(df)
        df = organize_columns(df)


core           INFO 	Loading data for São Paulo Grand Prix - Qualifying [v3.7.0]
2026-03-11 19:59:06,357 - INFO - Loading data for São Paulo Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
2026-03-11 19:59:06,358 - INFO - Using cached data for session_info
req            INFO 	Using cached data for driver_info


2026-03-11 19:59:06,359 - INFO - Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
2026-03-11 19:59:06,384 - INFO - Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
2026-03-11 19:59:06,390 - INFO - Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
2026-03-11 19:59:06,393 - INFO - Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
2026-03-11 19:59:06,393 - INFO - Using cached data for timing_app_data
core           INFO 	Processing timing data...
2026-03-11 19:59:06,393 - INFO - Processing timing data...
core        WARNING 	No lap data for driver 5
2026-03-11 19:59:06,891 - WARNING - No lap data for driver 5
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 5)
2026-03-11 19:59:07,023 - WARNING - Failed to perform lap accuracy check - 

In [95]:
df

,Driver,Sector1Time,Sector2Time,Sector3Time,IdealLap,DiffLap,LapTimeSec,IdealLapSec,DiffLapSec,Race,RaceNumber
0,NOR,00:00:17.998000,00:00:35.534000,00:00:15.941000,00:01:09.473000,00:00:00.038000,69.511,69.473,0.038,São Paulo Grand Prix,21
1,ANT,00:00:18.035000,00:00:35.582000,00:00:16.038000,00:01:09.473000,00:00:00.212000,69.685,69.473,0.212,São Paulo Grand Prix,21
2,BEA,00:00:18.062000,00:00:35.641000,00:00:15.994000,00:01:09.473000,00:00:00.282000,69.755,69.473,0.282,São Paulo Grand Prix,21
3,LEC,00:00:18.012000,00:00:35.680000,00:00:16.034000,00:01:09.473000,00:00:00.328000,69.801,69.473,0.328,São Paulo Grand Prix,21
4,PIA,00:00:18.127000,00:00:35.596000,00:00:16.030000,00:01:09.473000,00:00:00.362000,69.835,69.473,0.362,São Paulo Grand Prix,21
5,GAS,00:00:18.075000,00:00:35.643000,00:00:15.974000,00:01:09.473000,00:00:00.384000,69.857,69.473,0.384,São Paulo Grand Prix,21
6,RUS,00:00:18.081000,00:00:35.748000,00:00:16.033000,00:01:09.473000,00:00:00.407000,69.880,69.473,0.407,São Paulo Grand Prix,21
7,HAD,00:00:18.086000,00:00:35.690000,00:00:15.981000,00:01:09.473000,00:00:00.458000,69.931,69.473,0.458,São Paulo Grand Prix,21
8,LAW,00:00:18.164000,00:00:35.677000,00:00:15.984000,00:01:09.473000,00:00:00.477000,69.950,69.473,0.477,São Paulo Grand Prix,21
9,HUL,00:00:18.183000,00:00:35.680000,00:00:15.954000,00:01:09.473000,00:00:00.512000,69.985,69.473,0.512,São Paulo Grand Prix,21
